# 02.4 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/02-regression/04-gradient-descent/solutions.ipynb)

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd
five = pd.read_csv(data_path("shopwave_visits_spend_5.csv"))
five

import numpy as np, pandas as pd
x = five["visits_last_month"].to_numpy(dtype=float)
y = five["spend_next_month"].to_numpy(dtype=float)

def gradient_descent(x, y, alpha, steps, m=0.0, c=0.0):
    """Return the path of (m, c, loss) for `steps` iterations."""
    path = []
    for _ in range(steps):
        y_hat = m * x + c
        resid = y - y_hat
        loss = (resid**2).mean()
        path.append((m, c, loss))
        grad_m = -2 * (x * resid).mean()
        grad_c = -2 * resid.mean()
        m -= alpha * grad_m
        c -= alpha * grad_c
    return pd.DataFrame(path, columns=["m", "c", "loss"])

## Exercise 1

$\hat y = 10x + 30$: 50, 70, 90, 110, 130. Residuals: 0, 10, 0, 20, 20 (sum 50). $x \cdot r$: 0, 40, 0, 160, 200 (sum 400).

$\partial J/\partial m = -\frac{2}{5}(400) = -160$, $\ \partial J/\partial c = -\frac{2}{5}(50) = -20$.

$m = 10 + 1.6 = 11.6$, $\ c = 30 + 0.2 = 30.2$.

Both go **up**: every residual is ≥ 0, so predictions are too low on average, so the gradient is negative and the update raises both knobs. ($c$ will later have to come back down from 30 to 25; gradient descent does not know the destination, only the local slope.)

In [2]:
p = gradient_descent(x, y, alpha=0.01, steps=2, m=10.0, c=30.0)
print(p.round(3))

      m     c    loss
0  10.0  30.0  180.00
1  11.6  30.2   36.52


## Exercise 2

In [3]:
for alpha in [0.001, 0.005, 0.01, 0.02, 0.025, 0.03]:
    with np.errstate(over="ignore", invalid="ignore"):
        p = gradient_descent(x, y, alpha=alpha, steps=2000)
    m_f, c_f, loss_f = p.iloc[-1]
    if np.isfinite(loss_f):
        print(f"α={alpha:<6} final loss {loss_f:>10.4g}   m={m_f:.3f} c={c_f:.3f}")
    else:
        print(f"α={alpha:<6} DIVERGED (loss overflowed to inf/nan)")

α=0.001  final loss       52.8   m=14.033 c=13.803
α=0.005  final loss      30.08   m=12.588 c=24.357
α=0.01   final loss         30   m=12.502 c=24.982
α=0.02   final loss         30   m=12.500 c=25.000
α=0.025  DIVERGED (loss overflowed to inf/nan)
α=0.03   DIVERGED (loss overflowed to inf/nan)


Up to about 0.02 the walk converges (faster as α grows); at 0.025 and beyond it diverges. The edge depends on the data's scale: for this bowl it is roughly α ≈ 0.023. There is no universal "good" learning rate, only a good one for a given loss surface.

## Exercise 3

In [4]:
m, c, alpha, prev = 0.0, 0.0, 0.01, np.inf
for step in range(1, 100_000):
    resid = y - (m * x + c)
    loss = (resid**2).mean()
    if prev - loss < 1e-6:
        break
    prev = loss
    m -= alpha * (-2 * (x * resid).mean())
    c -= alpha * (-2 * resid.mean())
print(f"stopped after {step} steps: m={m:.4f}, c={c:.4f}, loss={loss:.4f}")

stopped after 1880 steps: m=12.5038, c=24.9724, loss=30.0001


Just under two thousand steps, and $c$ is still slightly short of 25 when the improvement per step drops below the threshold. The stopping rule trades a tiny bit of accuracy for not running forever; every ML library has one (often called `tol`).

## Exercise 4

In [5]:
xs = (x - x.mean()) / x.std()
p = gradient_descent(xs, y, alpha=0.1, steps=500)
p["m_orig"] = p["m"] / x.std()
p["c_orig"] = p["c"] - p["m_orig"] * x.mean()
close = p[(p["m_orig"].sub(12.5).abs() < 0.01) & (p["c_orig"].sub(25).abs() < 0.01)]
print("first step within 0.01 of (12.5, 25):", close.index[0])
print(p[["m_orig", "c_orig", "loss"]].iloc[[0, 10, 50, int(close.index[0])]].round(3))

first step within 0.01 of (12.5, 25): 36
    m_orig  c_orig       loss
0    0.000   0.000  11280.000
10  11.158  22.316    159.704
50  12.500  25.000     30.000
36  12.496  24.992     30.001


From thousands of steps to a few dozen. Scaling turned the long narrow canyon into a round bowl, so one learning rate suits both knobs. Lesson 04.5 does this for every feature with `StandardScaler`.